In [1]:
import matplotlib
import matplotlib.pyplot as plt
import backtrader as bt
import pandas as pd

In [9]:
class MyBuySell(bt.observers.BuySell):
    plotlines = dict(
        buy=dict(marker='^', markersize=8.0, color='blue', fillstyle='full'),
        sell=dict(marker='v', markersize=8.0, color='red', fillstyle='full')
    )

In [10]:
class RsiSignalStrategy(bt.SignalStrategy):
    params = dict(rsi_periods=14, rsi_upper=70, rsi_lower=30, rsi_mid=50)

    def __init__(self):

        # 상대 강도 지수(RSI) : 계산기간(period), 과매수 기준(upperband), 과매도 기준(lowerband)
        rsi = bt.indicators.RSI(period=self.p.rsi_periods,
                                upperband=self.p.rsi_upper,
                                lowerband=self.p.rsi_lower)

        
        # 참고용 상대 강도 지수(RSI)
        #bt.talib.RSI(self.data, plotname='TA_RSI')

        # long 신호
        rsi_signal_long = bt.ind.CrossUp(rsi, self.p.rsi_lower, plot=False)
        self.signal_add(bt.SIGNAL_LONG, rsi_signal_long)
        self.signal_add(bt.SIGNAL_LONGEXIT, -(rsi > self.p.rsi_mid))

        # Short 신호
        rsi_signal_short = -bt.ind.CrossDown(rsi, self.p.rsi_upper, plot=False)
        self.signal_add(bt.SIGNAL_SHORT, rsi_signal_short)
        self.signal_add(bt.SIGNAL_SHORTEXIT, rsi < self.p.rsi_mid)

### 백테스트용 데이터 불러오기(META 2018년도)

In [11]:
dir_nm = "dailyStock"
target = "META"
file_path = f"{dir_nm}/{target}.csv"

meta_df = pd.read_csv(file_path, encoding="utf-8")
meta_df['Date'] = pd.to_datetime(meta_df['Date'])
meta_df = meta_df.set_index("Date")

meta_df = meta_df.loc["2018-01-01":"2018-12-31"]

data = bt.feeds.PandasData(dataname=meta_df)

### 백테스트 설정

In [12]:
cerebro = bt.Cerebro(stdstats = False)

cerebro.addstrategy(RsiSignalStrategy)
cerebro.adddata(data)
cerebro.broker.setcash(1000.0)
cerebro.broker.setcommission(commission=0.001)
cerebro.addobserver(MyBuySell)
cerebro.addobserver(bt.observers.Value)

### 백테스트 실행

In [13]:
print(f'포트폴리오 시작가: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'포트폴리오 종료가: {cerebro.broker.getvalue():.2f}')

포트폴리오 시작가: 1000.00
포트폴리오 종료가: 1004.47


### 백테스트 결과 도식화

In [14]:
cerebro.plot(iplot=False, volume=False)

[[<Figure size 640x480 with 3 Axes>]]